In [37]:
"""Data preprocessing and synthetic collage generation utilities."""

from __future__ import annotations

import os
import random
import shutil
from pathlib import Path
from typing import Sequence

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


In [61]:
def remove_excess_files(dataset_dir: str | Path = "Dataset", target_count: int = 80) -> dict[str, int]:
    """Prune excess images so each subject directory contains at most target_count files.

    Args:
        dataset_dir: Path to root dataset directory containing per-subject folders.
        target_count: Maximum number of images retained per class (default: 80).

    Returns:
        Dictionary mapping subject names to the number of deleted excess files.
    """
    root = Path(dataset_dir).resolve()
    deleted_counts: dict[str, int] = {}
    for subject_dir in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        files = sorted(f for f in subject_dir.iterdir() if f.is_file() and not f.name.startswith("."))
        removed = 0
        if len(files) > target_count:
            for excess_file in files[target_count:]:
                excess_file.unlink()
                removed += 1
        deleted_counts[subject_dir.name] = removed
    return deleted_counts


def rename_files_consistently(dataset_dir: str | Path = "Dataset") -> None:
    """Rename image files uniformly to the format '{subject}_{index}.jpg'.

    Args:
        dataset_dir: Path to root dataset directory containing per-subject folders.
    """
    root = Path(dataset_dir).resolve()
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    for subject_dir in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        name = subject_dir.name
        clean_name = name[5:] if name.startswith("pins_") else name
        files = sorted(f for f in subject_dir.iterdir() if f.is_file() and f.suffix.lower() in valid_exts)
        temp_files: list[Path] = []
        for idx, f in enumerate(files):
            temp_path = subject_dir / f"__temp_{idx}_{f.name}"
            f.rename(temp_path)
            temp_files.append(temp_path)
        for idx, temp_path in enumerate(temp_files):
            target_path = subject_dir / f"{clean_name}_{idx}.jpg"
            temp_path.rename(target_path)


def inspect_dataset_file_counts(dataset_dir: str | Path = "Dataset") -> dict[str, int]:
    """List and return the image count for each subject directory.

    Args:
        dataset_dir: Path to root dataset directory.

    Returns:
        Dictionary mapping subject name to file count.
    """
    root = Path(dataset_dir).resolve()
    counts: dict[str, int] = {}
    for subject_dir in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        img_count = sum(1 for f in subject_dir.iterdir() if f.is_file() and not f.name.startswith("."))
        counts[subject_dir.name] = img_count
        print(f"{subject_dir.name:25s}: {img_count} images")
    if counts:
        print(f"\nTotal Subjects: {len(counts)} | Min count: {min(counts.values())} | Max count: {max(counts.values())}")
    return counts


def create_full_data(
    dataset_dir: str | Path = "Dataset", 
    destination_dir: str | Path = "Full Data"
) -> Path:
    """Consolidate copies of all subject images into a flat unified directory.

    Args:
        dataset_dir: Source dataset directory containing subject subfolders.
        destination_dir: Destination flat folder to store all image copies.

    Returns:
        Path to the populated destination directory.
    """
    src_root = Path(dataset_dir).resolve()
    dst_root = Path(destination_dir).resolve()
    dst_root.mkdir(parents=True, exist_ok=True)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    copied = 0
    for subject_dir in sorted(p for p in src_root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        for img_file in subject_dir.iterdir():
            if img_file.is_file() and img_file.suffix.lower() in valid_exts:
                dst_file = dst_root / img_file.name
                shutil.copyfile(img_file, dst_file)
                copied += 1
    print(f"Copied {copied} images to {dst_root}")
    return dst_root


def resize_dataset(
    dataset_dir: str | Path = "Dataset", 
    target_size: tuple[int, int] = (128, 128)
) -> None:
    """Resize all dataset images in-place to target resolution with proper resource management.

    Args:
        dataset_dir: Path to root dataset directory.
        target_size: (width, height) resolution.
    """
    root = Path(dataset_dir).resolve()
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    resized_count = 0
    for subject_dir in sorted(p for p in root.iterdir() if p.is_dir() and not p.name.startswith(".")):
        for img_file in subject_dir.iterdir():
            if img_file.is_file() and img_file.suffix.lower() in valid_exts:
                with Image.open(img_file) as im:
                    if im.size != target_size:
                        im_resized = im.resize(target_size, Image.Resampling.LANCZOS)
                        im_resized.save(img_file)
                        resized_count += 1
    print(f"Resized {resized_count} images to {target_size}")


def collage_maker(
    dataset_dir: str | Path = "Dataset",
    collages_dir: str | Path = "Collages",
    num_collages: int = 80,
    grid_dim: int = 4,
    tile_size: int = 128,
    seed: int = 42
) -> Path:
    """Synthesize multi-face grid collages using images of enrolled subjects in random cells.

    Cross-platform path handling and robust array composition without os.chdir mutations.

    Args:
        dataset_dir: Path to dataset folder containing subject folders.
        collages_dir: Output folder to store synthetic collages.
        num_collages: Number of collages to generate.
        grid_dim: Dimension of square grid (4 -> 4x4 = 16 subjects).
        tile_size: Width and height of each face tile in pixels.
        seed: Random seed for reproducible permutations.

    Returns:
        Path to directory containing synthesized collages.
    """
    src_root = Path(dataset_dir).resolve()
    dst_root = Path(collages_dir).resolve()
    dst_root.mkdir(parents=True, exist_ok=True)
    rng = random.Random(seed)
    subject_dirs = sorted([d.name for d in src_root.iterdir() if d.is_dir() and not d.name.startswith(".")])
    cells_needed = grid_dim * grid_dim
    if len(subject_dirs) < cells_needed:
        raise ValueError(f"Need at least {cells_needed} subjects for {grid_dim}x{grid_dim} collage, found {len(subject_dirs)}")
    for i in range(num_collages):
        chosen_subjects = list(subject_dirs)
        rng.shuffle(chosen_subjects)
        v_rows: list[np.ndarray] = []
        for row_idx in range(grid_dim):
            h_cols: list[np.ndarray] = []
            for col_idx in range(grid_dim):
                subject_name = chosen_subjects[row_idx * grid_dim + col_idx]
                tile_path = src_root / subject_name / f"{subject_name}_{i}.jpg"
                if not tile_path.is_file():
                    candidates = list((src_root / subject_name).glob("*.jpg"))
                    tile_path = candidates[0] if candidates else None
                if tile_path is not None and tile_path.is_file():
                    tile_img = cv2.imread(str(tile_path))
                    if tile_img is not None and (tile_img.shape[1] != tile_size or tile_img.shape[0] != tile_size):
                        tile_img = cv2.resize(tile_img, (tile_size, tile_size))
                else:
                    tile_img = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                h_cols.append(tile_img)
            v_rows.append(cv2.hconcat(h_cols))
        collage_img = cv2.vconcat(v_rows)
        dest_path = dst_root / f"Collage_{i}.jpg"
        cv2.imwrite(str(dest_path), collage_img)
    print(f"Successfully verified/generated collages in: {dst_root}")
    return dst_root

# Backward compatibility aliases
remove_some_files = remove_excess_files
remane_correctly = rename_files_consistently
rename_correctly = rename_files_consistently
number_of_files = inspect_dataset_file_counts
Create_full_data = create_full_data


In [60]:
# Verify current dataset and collages status
file_counts = inspect_dataset_file_counts("Dataset")
collages_count = len(list(Path("Collages").glob("*.jpg")))
print(f"Existing collages in 'Collages/': {collages_count}")

# If collages do not exist, run collage_maker() to synthesize them
if collages_count == 0:
    collage_maker(num_collages=5)
